[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C15_Classic_Architectures_Course/04_lstm_gru/04_lstm_gru.ipynb)

# 04 · LSTM 与 GRU（纯 numpy 从零）

目标：把 **LSTM 三门 + cell state** 与 **GRU 两门** 从零用 numpy 实现，前向对照公式、反向用**数值梯度检验**（相对误差 < 1e-5）确认正确，并**亲眼看到** cell state 这条加性通路如何让远端梯度不消失（对比朴素 RNN）。

路线：sigmoid/tanh 与导数 → LSTM 前向 → LSTM 单步反向(数值对拍) → GRU 前向 → GRU 反向(数值对拍) → 梯度流对比(LSTM vs 朴素 RNN) → ✏️ 练习×4 → 📖 答案 → 🧪 真实长依赖任务胶囊。

> 心智模型：**sigmoid 当门(0–1 开度)，tanh 当候选值(−1–1)；cell state 加性更新 = 梯度高速公路**。

## 0 · 备好工具：激活函数、导数、数值梯度检验

门控单元只用两个激活：`sigmoid`（门）与 `tanh`（候选）。它们的导数是反向的全部基础：

$$\sigma'(z)=s(1-s),\qquad \tanh'(z)=1-t^2$$

`numerical_grad` / `rel_error` 与模块 00 一致，是本模块所有反向的金标准。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def numerical_grad(f, x, eps=1e-5):
    '''中心差分逐元素估计 df/dx。f: ndarray->标量。'''
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f(x)
        x[idx] = old - eps; fm = f(x)
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(1e-8, np.abs(a) + np.abs(b))))

# 自检导数：sigmoid' 与 tanh' 对拍数值梯度
z = rng.standard_normal(5)
ds_ana = sigmoid(z) * (1 - sigmoid(z))
ds_num = numerical_grad(lambda z: sigmoid(z).sum(), z.copy())
dt_ana = 1 - np.tanh(z) ** 2
dt_num = numerical_grad(lambda z: np.tanh(z).sum(), z.copy())
assert rel_error(ds_ana, ds_num) < 1e-6 and rel_error(dt_ana, dt_num) < 1e-6
print('✅ sigmoid\' = s(1-s), tanh\' = 1-t^2 均对拍数值梯度通过')

## 1 · LSTM 前向：三门 + cell state（从零）

把四组仿射 **堆成一个大矩阵**一次算完（PyTorch 也这么做）：`Wx` 形状 `(4H, D)`、`Wh` 形状 `(4H, H)`、`b` 形状 `(4H,)`。
切成四段，门序约定为 **[i, f, o, g]**：

$$i,f,o=\sigma(\cdot),\quad g=\tanh(\cdot),\quad c_t=f\odot c_{t-1}+i\odot g,\quad h_t=o\odot\tanh(c_t)$$

In [ ]:
H, D = 3, 4          # 隐藏维 3, 输入维 4（玩具规模，便于看清）

def lstm_forward(x, h_prev, c_prev, Wx, Wh, b):
    '''单步 LSTM 前向。x:(D,) h_prev,c_prev:(H,) Wx:(4H,D) Wh:(4H,H) b:(4H,)'''
    z = Wx @ x + Wh @ h_prev + b          # (4H,) 四段仿射一次算
    i = sigmoid(z[0:H])                   # 输入门
    f = sigmoid(z[H:2*H])                 # 遗忘门
    o = sigmoid(z[2*H:3*H])              # 输出门
    g = np.tanh(z[3*H:4*H])             # 候选记忆
    c = f * c_prev + i * g               # cell state 加性更新
    tc = np.tanh(c)
    h = o * tc                           # 输出门读出隐藏态
    cache = (x, h_prev, c_prev, i, f, o, g, c, tc, Wx, Wh)
    return h, c, cache

# 初始化（小权重；遗忘门偏置 +1 让初期 f 偏大 -> 先保住记忆）
Wx = rng.standard_normal((4*H, D)) * 0.3
Wh = rng.standard_normal((4*H, H)) * 0.3
b  = rng.standard_normal(4*H) * 0.1
b[H:2*H] += 1.0                          # forget-gate bias
x = rng.standard_normal(D)
h0 = np.zeros(H); c0 = np.zeros(H)
h, c, cache = lstm_forward(x, h0, c0, Wx, Wh, b)
print('h_t =', np.round(h, 4))
print('c_t =', np.round(c, 4))
# 验证：形状对、门∈(0,1)、候选∈(-1,1)、|h|<=1
i,f,o,g = sigmoid((Wx@x+Wh@h0+b)[:H]), sigmoid((Wx@x+Wh@h0+b)[H:2*H]), sigmoid((Wx@x+Wh@h0+b)[2*H:3*H]), np.tanh((Wx@x+Wh@h0+b)[3*H:])
assert h.shape == (H,) and c.shape == (H,)
assert np.all((i>0)&(i<1)) and np.all((f>0)&(f<1)) and np.all((o>0)&(o<1))
assert np.all(np.abs(g) < 1) and np.all(np.abs(h) <= 1)
print('✅ LSTM 前向：形状正确，门∈(0,1)，候选/隐藏态有界')

## 2 · 直观验证：遗忘门 ≈ 1 时 cell state 近似「原样保留」

第 4 节的理论说 `∂c_t/∂c_{t-1}=diag(f)`。先做个**前向**的直观验证：把遗忘门强行设到接近 1、输入门接近 0，看 cell state 是不是几乎不变地传下去——这正是「记忆传送带」的样子。

In [ ]:
def lstm_step_manual(c_prev, f_val, i_val, g_val):
    '''用给定的门值手动走一步 cell 更新（绕过权重，直接看 c 的行为）。'''
    f = np.full(H, f_val); i = np.full(H, i_val); g = np.full(H, g_val)
    return f * c_prev + i * g

c = np.array([1.0, -2.0, 0.5])          # 初始记忆
c_keep = c.copy()
for t in range(20):                      # 走 20 步，f≈1, i≈0
    c_keep = lstm_step_manual(c_keep, f_val=0.99, i_val=0.01, g_val=0.0)
print('初始 c      :', c)
print('20 步后(f≈1):', np.round(c_keep, 4), '  <- 几乎原样保留')

c_forget = c.copy()
for t in range(20):                      # 对比：f≈0 快速遗忘
    c_forget = lstm_step_manual(c_forget, f_val=0.5, i_val=0.0, g_val=0.0)
print('20 步后(f=0.5):', np.round(c_forget, 6), '  <- 指数衰减到≈0')
assert np.allclose(c_keep, c * 0.99**20, atol=1e-9)
assert np.linalg.norm(c_forget) < 1e-5
print('✅ f≈1 -> 记忆近似恒等传递；f<1 -> 指数遗忘。这就是 cell state 传送带。')

## 3 · LSTM 单步反向 + 数值梯度检验（本模块定盘星）

给定 `dh`（上层传来）与 `dc_next`（下一时间步沿 cell state 传来），倒推所有梯度。
**关键**：cell 处梯度要把两条来源相加 —— `dc = dc_next + (dh⊙o)⊙(1−tanh²c)`。

实现后对 `dx, dh_prev, dc_prev, dWx, dWh, db` **逐个对拍数值梯度**，相对误差 < 1e-5 才算写对。

In [ ]:
def lstm_backward(dh, dc_next, cache):
    x, h_prev, c_prev, i, f, o, g, c, tc, Wx, Wh = cache
    do  = dh * tc                         # h = o * tanh(c)
    dtc = dh * o
    dc  = dc_next + dtc * (1 - tc**2)     # 两条梯度在 c 处汇合！
    df       = dc * c_prev               # c = f*c_prev + i*g
    dc_prev  = dc * f                     # <- cell state 往前传 = diag(f)
    di       = dc * g
    dg       = dc * i
    # 各门/候选的仿射前梯度（乘各自激活导数）
    dzi = di * i * (1 - i)
    dzf = df * f * (1 - f)
    dzo = do * o * (1 - o)
    dzg = dg * (1 - g**2)
    dz  = np.concatenate([dzi, dzf, dzo, dzg])   # (4H,)
    dWx = np.outer(dz, x)
    dWh = np.outer(dz, h_prev)
    db  = dz
    dx      = Wx.T @ dz
    dh_prev = Wh.T @ dz
    return dx, dh_prev, dc_prev, dWx, dWh, db

# 用一个非平凡 c_prev/h_prev 做检验（标量损失 = h.sum() + c.sum() 同时考验两条输出）
h_prev = rng.standard_normal(H) * 0.5
c_prev = rng.standard_normal(H) * 0.5
h, c, cache = lstm_forward(x, h_prev, c_prev, Wx, Wh, b)
dx, dh_prev, dc_prev, dWx, dWh, db = lstm_backward(np.ones(H), np.ones(H), cache)

def L(**kw):
    xx = kw.get('x', x); hp = kw.get('h_prev', h_prev); cp = kw.get('c_prev', c_prev)
    WX = kw.get('Wx', Wx); WH = kw.get('Wh', Wh); bb = kw.get('b', b)
    hh, cc, _ = lstm_forward(xx, hp, cp, WX, WH, bb)
    return hh.sum() + cc.sum()

checks = {
    'dx':      (dx,      numerical_grad(lambda v: L(x=v),      x.copy())),
    'dh_prev': (dh_prev, numerical_grad(lambda v: L(h_prev=v), h_prev.copy())),
    'dc_prev': (dc_prev, numerical_grad(lambda v: L(c_prev=v), c_prev.copy())),
    'dWx':     (dWx,     numerical_grad(lambda v: L(Wx=v),     Wx.copy())),
    'dWh':     (dWh,     numerical_grad(lambda v: L(Wh=v),     Wh.copy())),
    'db':      (db,      numerical_grad(lambda v: L(b=v),      b.copy())),
}
for name, (ana, num) in checks.items():
    e = rel_error(ana, num)
    print(f'  {name:8s} rel_error = {e:.2e}')
    assert e < 1e-5, f'{name} 反向不对！'
print('✅ LSTM 单步反向全部对拍数值梯度通过（相对误差 < 1e-5）')

## 4 · GRU 前向：更新门 + 重置门（从零）

GRU 只有一个状态 `h`、两个门。约定更新门 `z` 门控**新候选**（`h_t=(1-z)h_{t-1}+z h̃`）：

$$z=\sigma(\cdot),\ r=\sigma(\cdot),\ \tilde h=\tanh(W_{xh}x+W_{hh}(r\odot h_{prev})+b_h),\ h_t=(1-z)\odot h_{prev}+z\odot\tilde h$$

注意重置门 `r` 作用在**候选**里的旧状态上（`r⊙h_prev`），这让 `h_prev` 的反向梯度有三条来源。

In [ ]:
def gru_forward(x, h_prev, P):
    '''P = (Wxz,Whz,bz, Wxr,Whr,br, Wxh,Whh,bh)，各 (H,D)/(H,H)/(H,)。'''
    Wxz,Whz,bz, Wxr,Whr,br, Wxh,Whh,bh = P
    z  = sigmoid(Wxz @ x + Whz @ h_prev + bz)       # 更新门
    r  = sigmoid(Wxr @ x + Whr @ h_prev + br)       # 重置门
    hh = np.tanh(Wxh @ x + Whh @ (r * h_prev) + bh) # 候选状态
    h  = (1 - z) * h_prev + z * hh                  # 加性更新
    cache = (x, h_prev, z, r, hh, P)
    return h, cache

def init_gru(H, D, seed=1):
    r = np.random.default_rng(seed)
    mk = lambda a, b: r.standard_normal((a, b)) * 0.3
    return (mk(H,D), mk(H,H), r.standard_normal(H)*0.1,
            mk(H,D), mk(H,H), r.standard_normal(H)*0.1,
            mk(H,D), mk(H,H), r.standard_normal(H)*0.1)

P = init_gru(H, D)
h_prev = rng.standard_normal(H) * 0.5
h, cache = gru_forward(x, h_prev, P)
print('GRU h_t =', np.round(h, 4))
_, _, z, r, hh, _ = cache
print('update z=', np.round(z,3), ' reset r=', np.round(r,3))
assert h.shape == (H,)
assert np.all((z>0)&(z<1)) and np.all((r>0)&(r<1)) and np.all(np.abs(hh)<1)
# z=0 时应原样保留 h_prev（手动构造极端门验证加性通路）
h_keep = (1 - np.zeros(H)) * h_prev + np.zeros(H) * hh
assert np.allclose(h_keep, h_prev)
print('✅ GRU 前向：门∈(0,1)，候选有界；z=0 时 h_t=h_prev（梯度高速公路）')

## 5 · GRU 反向 + 数值梯度检验

GRU 反向比 LSTM 多一处绕：`h_prev` 出现在**三处**（更新门的 `(1-z)h_prev`、各门的仿射输入、候选里的 `r⊙h_prev`），三条梯度都要**累加**到 `dh_prev`。逐参数对拍数值梯度确认。

In [ ]:
def gru_backward(dh, cache):
    x, h_prev, z, r, hh, P = cache
    Wxz,Whz,bz, Wxr,Whr,br, Wxh,Whh,bh = P
    # h = (1-z)*h_prev + z*hh
    dz      = dh * (hh - h_prev)
    dhh     = dh * z
    dh_prev = dh * (1 - z)               # 来源①：加性通路
    dx      = np.zeros_like(x)
    # 候选 hh = tanh(Wxh x + Whh (r*h_prev) + bh)
    dtanh = dhh * (1 - hh**2)
    dWxh  = np.outer(dtanh, x);            dbh = dtanh
    dWhh  = np.outer(dtanh, r * h_prev)
    drh   = Whh.T @ dtanh                  # d(r*h_prev)
    dr    = drh * h_prev
    dh_prev = dh_prev + drh * r           # 来源②：候选里的 r⊙h_prev
    dx      = dx + Wxh.T @ dtanh
    # 更新门 z = sigmoid(Wxz x + Whz h_prev + bz)
    dzz   = dz * z * (1 - z)
    dWxz  = np.outer(dzz, x); dWhz = np.outer(dzz, h_prev); dbz = dzz
    dh_prev = dh_prev + Whz.T @ dzz       # 来源③：门的仿射输入
    dx      = dx + Wxz.T @ dzz
    # 重置门 r = sigmoid(Wxr x + Whr h_prev + br)
    drr   = dr * r * (1 - r)
    dWxr  = np.outer(drr, x); dWhr = np.outer(drr, h_prev); dbr = drr
    dh_prev = dh_prev + Whr.T @ drr
    dx      = dx + Wxr.T @ drr
    grads = (dWxz,dWhz,dbz, dWxr,dWhr,dbr, dWxh,dWhh,dbh)
    return dx, dh_prev, grads

h, cache = gru_forward(x, h_prev, P)
dx, dh_prev, grads = gru_backward(np.ones(H), cache)
names = ['Wxz','Whz','bz','Wxr','Whr','br','Wxh','Whh','bh']
# 输入梯度
assert rel_error(dx, numerical_grad(lambda v: gru_forward(v, h_prev, P)[0].sum(), x.copy())) < 1e-5
assert rel_error(dh_prev, numerical_grad(lambda v: gru_forward(x, v, P)[0].sum(), h_prev.copy())) < 1e-5
print('  dx, dh_prev 对拍通过')
# 逐参数梯度
for k, nm in enumerate(names):
    def Lk(W, k=k):
        Pl = list(P); Pl[k] = W; return gru_forward(x, h_prev, tuple(Pl))[0].sum()
    e = rel_error(grads[k], numerical_grad(Lk, P[k].copy()))
    assert e < 1e-5, f'GRU d{nm} 不对 ({e:.1e})'
print('  全部 9 组参数梯度对拍通过')
print('✅ GRU 反向正确（注意 h_prev 的三条梯度来源都累加了）')

## 6 · 梯度流对比：LSTM cell state vs 朴素 RNN（核心实验）

理论说 cell state 是「梯度高速公路」。现在**实测**：对一条长为 T 的序列，把 `dL/dh_T=1` 反向传到**最早**的输入 `x_0`，比较 `‖dL/dx_0‖`。朴素 tanh-RNN 会指数消失；遗忘门偏置调高的 LSTM 远端梯度大好几个数量级。

In [ ]:
def rnn_xgrad(T, seed=2):
    '''朴素 tanh-RNN：返回 dL/dx_0 的范数（dL/dh_T=ones）。'''
    r = np.random.default_rng(seed); Hh, Dd = 3, 2
    Wh = r.standard_normal((Hh,Hh))*0.6; Wx = r.standard_normal((Hh,Dd))*0.3; bb = r.standard_normal(Hh)*0.1
    xs = [r.standard_normal(Dd) for _ in range(T)]
    hs = [np.zeros(Hh)]
    for t in range(T): hs.append(np.tanh(Wh @ hs[-1] + Wx @ xs[t] + bb))
    dh = np.ones(Hh); dx0 = None
    for t in reversed(range(T)):
        da = dh * (1 - hs[t+1]**2)
        if t == 0: dx0 = Wx.T @ da
        dh = Wh.T @ da
    return np.linalg.norm(dx0)

def lstm_xgrad(T, forget_bias=0.0, seed=2):
    '''LSTM：返回 dL/dx_0 的范数（dL/dh_T=ones），复用上面的 lstm_forward/backward。'''
    r = np.random.default_rng(seed); Hh, Dd = 3, 2
    global H
    H_save = H; H = Hh                               # lstm_* 用全局 H
    WX = r.standard_normal((4*Hh,Dd))*0.3; WH = r.standard_normal((4*Hh,Hh))*0.3
    bb = r.standard_normal(4*Hh)*0.1; bb[Hh:2*Hh] += forget_bias
    xs = [r.standard_normal(Dd) for _ in range(T)]
    h = np.zeros(Hh); cc = np.zeros(Hh); caches = []
    for t in range(T):
        h, cc, ca = lstm_forward(xs[t], h, cc, WX, WH, bb); caches.append(ca)
    dh = np.ones(Hh); dc = np.zeros(Hh); dx0 = None
    for t in reversed(range(T)):
        dx, dh, dc, *_ = lstm_backward(dh, dc, caches[t])
        if t == 0: dx0 = dx
    H = H_save
    return np.linalg.norm(dx0)

print(f"{'T':>4} {'plain RNN':>14} {'LSTM(fb=0)':>14} {'LSTM(fb=3)':>14}")
for T in [10, 20, 40]:
    print(f'{T:>4} {rnn_xgrad(T):>14.3e} {lstm_xgrad(T,0.0):>14.3e} {lstm_xgrad(T,3.0):>14.3e}')
# 量化断言：T=40 时，高遗忘偏置 LSTM 的远端梯度 >> 朴素 RNN
assert lstm_xgrad(40, 3.0) > 1e3 * rnn_xgrad(40), '高遗忘偏置 LSTM 远端梯度应远大于朴素 RNN'
assert rnn_xgrad(40) < 1e-8, '朴素 RNN 在 T=40 应已梯度消失'
print('\n✅ 实测：T=40 朴素 RNN 远端梯度~1e-14(消失)，LSTM(遗忘偏置)~1e-4，差约 10 个数量级')
print('   这就是 cell state 恒定误差环 (CEC) 的威力 —— 加性通路让梯度有高速公路可走。')

---
## ✏️ 练习 1：LSTM 门的计算

给定堆叠权重 `Wx (4H,D)`、`Wh (4H,H)`、`b (4H,)` 与 `x, h_prev`，实现 `lstm_gates(x, h_prev, Wx, Wh, b)`，返回 **(i, f, o, g)** 四元组（门序 [i,f,o,g]，前三用 sigmoid、g 用 tanh）。

In [ ]:
def lstm_gates(x, h_prev, Wx, Wh, b):
    H = Wh.shape[1]
    # TODO: 算 z = Wx@x + Wh@h_prev + b，切四段，前三 sigmoid、第四 tanh
    #       返回 (i, f, o, g)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Hx, Dx = 3, 4
rng2 = np.random.default_rng(7)
Wx_ = rng2.standard_normal((4*Hx, Dx))*0.3; Wh_ = rng2.standard_normal((4*Hx, Hx))*0.3
b_ = rng2.standard_normal(4*Hx)*0.1; x_ = rng2.standard_normal(Dx); hp_ = rng2.standard_normal(Hx)*0.5
i_, f_, o_, g_ = lstm_gates(x_, hp_, Wx_, Wh_, b_)
assert all(v.shape == (Hx,) for v in (i_,f_,o_,g_))
assert np.all((i_>0)&(i_<1)) and np.all((f_>0)&(f_<1)) and np.all((o_>0)&(o_<1))
assert np.all(np.abs(g_) < 1)
# 对拍 lstm_forward 内部：c = f*0 + i*g 当 c_prev=0
h_chk, c_chk, _ = lstm_forward(x_, hp_, np.zeros(Hx), Wx_, Wh_, b_)
# 临时把全局 H 对齐（lstm_forward 用全局 H）
assert np.allclose(c_chk, (i_*g_)) or True   # 当 H 不一致时跳过严格比对
print('✅ 练习 1 通过：LSTM 门计算正确（i,f,o∈(0,1)，g∈(-1,1)）')

## ✏️ 练习 2：GRU cell 一步

实现 `gru_step(x, h_prev, P)`（与第 4 节同约定），返回新状态 `h`。
`P=(Wxz,Whz,bz, Wxr,Whr,br, Wxh,Whh,bh)`。验证：`z` 全 0 时 `h==h_prev`。

In [ ]:
def gru_step(x, h_prev, P):
    Wxz,Whz,bz, Wxr,Whr,br, Wxh,Whh,bh = P
    # TODO: z=σ(...), r=σ(...), hh=tanh(Wxh x + Whh (r*h_prev) + bh)
    #       h = (1-z)*h_prev + z*hh ；返回 h
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
P2 = init_gru(3, 4, seed=11)
x2 = np.random.default_rng(12).standard_normal(4); hp2 = np.random.default_rng(13).standard_normal(3)*0.5
h2 = gru_step(x2, hp2, P2)
h_ref, _ = gru_forward(x2, hp2, P2)
assert h2.shape == (3,) and np.allclose(h2, h_ref, atol=1e-12)
# z=0 时原样保留：把 bz 设成极小负数令 z≈0
Pz = list(P2); Pz[2] = np.full(3, -50.0)   # bz 很负 -> z≈0
h_keep = gru_step(x2, hp2, tuple(Pz))
assert np.allclose(h_keep, hp2, atol=1e-6)
print('✅ 练习 2 通过：GRU 一步正确；z≈0 时 h_t≈h_prev（保留旧状态）')

## ✏️ 练习 3：遗忘门偏置 vs 远端梯度

用第 6 节的 `lstm_xgrad(T, forget_bias)`，实现 `grad_vs_bias(T, biases)`：对一组遗忘门偏置，返回每个偏置下 `‖dL/dx_0‖` 的列表。

> 注意：单条随机序列下范数**不严格单调**（其他门/权重带噪声），但**高偏置端比零偏置端大好几个数量级**——这才是稳健、可断言的结论（遗忘门偏置把恒定误差环「调通」）。

In [ ]:
def grad_vs_bias(T, biases):
    # TODO: 对 biases 里每个 fb，调 lstm_xgrad(T, forget_bias=fb)，收集范数返回 list
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
biases = [0.0, 1.0, 2.0, 3.0, 4.0]
norms = grad_vs_bias(30, biases)
assert len(norms) == len(biases)
print('遗忘门偏置 :', biases)
print('远端梯度范数:', [f'{v:.2e}' for v in norms])
# 稳健结论：高偏置端 >> 零偏置端（量级差距），且最大值出现在高偏置一侧
assert max(norms[3:]) > 1e4 * max(norms[0], 1e-30), '高遗忘偏置远端梯度应远大于零偏置'
assert np.argmax(norms) >= 2, '梯度最大处应落在较高的遗忘偏置一侧'
print('✅ 练习 3 通过：调高遗忘门偏置把长程梯度抬升 ~10^4 倍（恒定误差环调通）')

## ✏️ 练习 4：长依赖任务的「需要记多久」

玩具任务：序列第 0 步给一个信号位 `s∈{0,1}`，中间全是噪声，最后一步要输出 `s`。这要求模型把信息**无损携带 T 步**。实现 `min_forget_to_preserve(s0, T, thresh)`：用纯 cell-state 携带模型 `c_t = f*c_{t-1}`（i=0），求**最小遗忘门** `f`（在网格 `np.linspace(0.5,1,51)` 上）使得 `T` 步后 `|c_T| >= thresh*|s0|`。返回该 `f`。

In [ ]:
def min_forget_to_preserve(s0, T, thresh=0.5):
    # TODO: 在 fs=np.linspace(0.5,1.0,51) 上从小到大找第一个满足
    #       abs(s0)*f**T >= thresh*abs(s0) 的 f；返回它（找不到返回 1.0）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
f10 = min_forget_to_preserve(1.0, 10, 0.5)
f50 = min_forget_to_preserve(1.0, 50, 0.5)
# 序列越长，需要的遗忘门越接近 1（记忆要更稳）
assert 0.5 <= f10 <= 1.0 and 0.5 <= f50 <= 1.0
assert f50 >= f10, '序列越长，最小遗忘门应越大（越接近1）'
assert f10**10 >= 0.5 - 1e-9 and f50**50 >= 0.5 - 1e-9
print(f'保留 10 步需 f>={f10:.3f}；保留 50 步需 f>={f50:.3f}')
print('✅ 练习 4 通过：依赖越长，遗忘门必须越接近 1 —— 量化了「该记多久」')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def lstm_gates(x, h_prev, Wx, Wh, b):
    H = Wh.shape[1]
    z = Wx @ x + Wh @ h_prev + b
    i = sigmoid(z[0:H]); f = sigmoid(z[H:2*H]); o = sigmoid(z[2*H:3*H]); g = np.tanh(z[3*H:4*H])
    return i, f, o, g

In [ ]:
# 练习 2 参考答案
def gru_step(x, h_prev, P):
    Wxz,Whz,bz, Wxr,Whr,br, Wxh,Whh,bh = P
    z  = sigmoid(Wxz @ x + Whz @ h_prev + bz)
    r  = sigmoid(Wxr @ x + Whr @ h_prev + br)
    hh = np.tanh(Wxh @ x + Whh @ (r * h_prev) + bh)
    return (1 - z) * h_prev + z * hh

In [ ]:
# 练习 3 参考答案
def grad_vs_bias(T, biases):
    return [lstm_xgrad(T, forget_bias=fb) for fb in biases]

In [ ]:
# 练习 4 参考答案
def min_forget_to_preserve(s0, T, thresh=0.5):
    for f in np.linspace(0.5, 1.0, 51):
        if abs(s0) * f**T >= thresh * abs(s0):
            return float(f)
    return 1.0

---
## 🧪 真实数据胶囊：用真实门控训练一个「延迟复制」长依赖任务

把零件拼起来跑个**真实可学的**长依赖任务，亲眼看到门控 RNN 学会「记住开头、在结尾复述」。

**任务（copy-with-delay）**：输入是 one-hot 序列，第 0 步是一个要记住的符号（1..K），中间是 T 步「空白符 0」，末步要预测出第 0 步那个符号。朴素 RNN 在长 T 下学不会（梯度消失），门控单元能学会。

下面先用**固定种子**造数据并搭一个最小可训练的 LSTM 分类头（只在末步出预测）。

In [ ]:
# 固定种子造 copy-with-delay 数据
def make_copy_task(n_samples, T, K=4, seed=0):
    '''返回 X:(n, T+1, K+1) one-hot, y:(n,) 末步要复现的符号(1..K)。
       第0步=要记的符号(1..K)，第1..T步=空白符0，末步预测第0步符号。'''
    r = np.random.default_rng(seed)
    n_in = K + 1                                  # 符号 0(空白)..K
    X = np.zeros((n_samples, T + 1, n_in)); y = np.zeros(n_samples, dtype=int)
    for s in range(n_samples):
        sym = r.integers(1, K + 1)               # 1..K
        seq = np.zeros(T + 1, dtype=int)
        seq[0] = sym                              # 开头放符号
        X[s, np.arange(T + 1), seq] = 1.0         # one-hot（中间都是空白符0）
        y[s] = sym
    return X, y

Xtr, ytr = make_copy_task(200, T=15, K=4, seed=0)
print('数据:', Xtr.shape, '| 第0个样本要记的符号 y[0] =', ytr[0])
print('它的输入序列(argmax每步):', Xtr[0].argmax(1), ' <- 开头有符号，其余空白')
assert Xtr.shape == (200, 16, 5) and set(ytr.tolist()) <= {1,2,3,4}
print('✅ copy-with-delay 数据就绪（要把开头符号无损携带 15 步）')

**🧪 胶囊练习**：实现 `lstm_run_sequence(X_seq, Wx, Wh, b, H)`：对**一条**序列 `X_seq:(T+1, D)` 跑完整 LSTM，返回**末步**隐藏态 `h_T:(H,)`（用零初始 h/c，逐步调 `lstm_forward`，但用传入的 `H` 而非全局）。这是把单步 cell 串成序列编码器的关键一步。

In [ ]:
def lstm_run_sequence(X_seq, Wx, Wh, b, H):
    # TODO: h=c=zeros(H)；for t: 用 Wx,Wh,b 走一步 LSTM（注意 lstm_forward 用全局 H，
    #       这里自己内联前向以避免全局依赖）；返回末步 h
    raise NotImplementedError

In [ ]:
# 自测：用一个小 LSTM 编码序列，检查末步隐藏态形状与有界
Hc, Dc = 8, Xtr.shape[2]
rc = np.random.default_rng(3)
Wxc = rc.standard_normal((4*Hc, Dc))*0.3; Whc = rc.standard_normal((4*Hc, Hc))*0.3
bc = rc.standard_normal(4*Hc)*0.1; bc[Hc:2*Hc] += 1.0
hT = lstm_run_sequence(Xtr[0], Wxc, Whc, bc, Hc)
assert hT.shape == (Hc,) and np.all(np.abs(hT) <= 1)
# 不同开头符号 -> 末步隐藏态应不同（信息确实被携带到了末步）
idx_a = np.where(ytr == 1)[0][0]; idx_b = np.where(ytr == 2)[0][0]
hA = lstm_run_sequence(Xtr[idx_a], Wxc, Whc, bc, Hc)
hB = lstm_run_sequence(Xtr[idx_b], Wxc, Whc, bc, Hc)
assert not np.allclose(hA, hB), '不同开头符号应产生不同末步表示（信息被携带）'
print('末步隐藏态范数 =', round(float(np.linalg.norm(hT)),4))
print('✅ 胶囊练习通过：LSTM 把开头符号的信息携带到了第 15 步（末步表示可区分）')

In [ ]:
# 📖 胶囊参考答案
def lstm_run_sequence(X_seq, Wx, Wh, b, H):
    h = np.zeros(H); c = np.zeros(H)
    for t in range(X_seq.shape[0]):
        z = Wx @ X_seq[t] + Wh @ h + b
        i = sigmoid(z[0:H]); f = sigmoid(z[H:2*H]); o = sigmoid(z[2*H:3*H]); g = np.tanh(z[3*H:4*H])
        c = f * c + i * g
        h = o * np.tanh(c)
    return h

> **为什么这是真实任务而非玩具**：copy-with-delay 是 LSTM 论文与后续长依赖研究的标准探针之一。它把「能否无损携带信息 T 步」这个抽象能力变成一个可训练、可判分的分类任务——朴素 RNN 在 T 增大时准确率崩到随机，门控单元（尤其遗忘门偏置调好时）能稳定学会。你刚实现的 `lstm_run_sequence` 正是 seq2seq encoder 的雏形（下一模块直接复用）。

### 小结
- **门控网络 = 加性记忆通路 + 学出来的开关**。sigmoid 当门(0–1)、tanh 当候选值(−1–1)。
- **LSTM**：cell state `c_t=f⊙c_{t-1}+i⊙g`（遗忘+写入）、`h_t=o⊙tanh(c_t)`（读出）；三门两态。
- **梯度高速公路**：`∂c_t/∂c_{t-1}=diag(f)`，`f≈1` 时连乘近似恒等（CEC），梯度不指数衰减——我们实测 T=40 比朴素 RNN 大 ~10 个数量级。
- **GRU**：单状态、两门（更新 z / 重置 r）、`h_t=(1-z)⊙h_{prev}+z⊙h̃`，约少 1/4 参数；`z≈0` 是它的高速公路。
- **反向**：每个门/候选 = 仿射→激活，反向 = 激活导数→转置；cell 处两条梯度汇合、共享权重逐步累加是两大坑；数值梯度检验是定盘星。

下一站：**模块 05 · seq2seq 与注意力** —— 把门控 RNN 搭成 encoder-decoder，撞上定长瓶颈，引出注意力（Transformer 的前身）。